# YES24에서 베스트셀러 정보 수직하기

In [16]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup as bs

In [17]:
url = "https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=120)
r = requests.get(url, params=payload)
print(r.url)
print(r.status_code)
response = bs(r.content, 'lxml')
response

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200


<!DOCTYPE html>
<html lang="ko">
<head><link href="https://m.yes24.com/Home/Best?DispNo=001" media="only screen and(max-width: 640px)" rel="alternate"/>
<meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
<meta content="text/html;charset=utf-8" http-equiv="Content-Type"/>
<meta content="dpr, width, viewport-width, rtt, downlink, ect, UA, UA-Platform, UA-Arch, UA-Model, UA-Mobile, UA-Full-Version" http-equiv="Accept-CH"/>
<meta content="86400" http-equiv="Accept-CH-Lifetime"/>
<meta content="unsafe-url" name="referrer"/>
<meta content="width=1170" name="viewport"/>
<title> 국내도서 종합 베스트 - 예스24 </title>
<meta content="예스24" name="title"/>
<meta content="도서, 티켓, 음반, 굿즈, 커뮤니티까지 문화 전반을 아우르는 국내 대표 문화콘텐츠 플랫폼, 아낌없는 혜택과 총알배송을 경험해보세요." name="description"/>
<meta content="인터넷 서점, 온라인 쇼핑, 상품 추천, 쇼핑몰, 상품 검색, 도서 정보, 국내도서, 외국도서, 전자책, eBook, 이북, 크레마, 공연, 콘서트, 뮤지컬, 음반, 예매, DVD, 블루레이, 예스24, YES24, 교보문고, 알라딘, 리센스, 예스24 도서용품, 친환경 PB 브랜드, 사은품, 굿즈" name="keywords"/>
<link href="//www.yes24.com/OpensearchDe

In [22]:
book_list = response.select("#yesBestList > li")
book_list

[<li class="" data-goods-no="166390503" data-iy-no="0" data-statgb="02">
 <div class="itemUnit">
 <div class="item_img">
 <div class="img_canvas">
 <div class="img_upper">
 <em class="ico rank">1</em>
 <span class="rank_info rank_even">
 <span class="ico"></span><em class="txt rank"></em><em class="txt">위 상승</em>
 </span>
 </div>
 <span class="img_item">
 <span class="img_grp">
 <a class="lnk_img" href="/product/goods/166390503" onclick="wiseLogV2('BS', '001_005_001', ''); ">
 <em class="img_bdr">
 <img alt="최소한의 삼국지" border="0" class="lazy" data-original="https://image.yes24.com/goods/166390503/L" src="https://image.yes24.com/momo/Noimg_L.jpg"/>
 </em>
 </a>
 </span>
 </span>
 </div>
 <div class="img_btn">
 <a class="btnC btn_preview" href="javascript:yes24GU.openPreviewCheck(166390503); wiseLogV2('BS', '001_005_011', '');"><span class="bWrap"><em class="txt">미리보기</em></span></a>
 </div>
 </div>
 <div class="item_info">
 <div class="info_row info_keynote">
 <span class="gd_keynote" id

In [23]:
len(book_list)

120

# li 1개는 1권의 정보
* 책 제목, 저자, 출판사, 출간일, 가격, 평점, 리뷰수

In [24]:
def text_clean(text):
    text = text.replace("\n", "").replace("\r", "").replace(",", "").replace("저", "").replace("역", "").strip()
    return text

In [25]:
# 책 제목
book_title = book_list[0].select_one(".gd_name").text

In [26]:
book_list[0].select_one(".authPub.info_auth").text

'\n최태성 저/이성원 감수\r\n                            '

In [28]:
text_clean(book_list[7].select_one(".authPub.info_auth").text)

'최태성'

In [30]:
# 저자와 역자가 같이 있을 때 저자
text_clean(book_list[1].select_one(".authPub.info_auth").text.split("/")[0])

'김난도 전미영 최지혜 권정윤 한다혜  외 7명                                        정보 더 보기'

In [31]:
# 저자와 역자가 같이 있을 때 역자
text_clean(book_list[1].select_one(".authPub.info_auth").text.split("/")[1])

'감추기김난도전미영최지혜권정윤한다혜이혜원이수진서유현전다현이준영이향은김나은'

In [32]:
# 저자가 1명일 때
author = text_clean(book_list[2].select_one(".authPub.info_auth").text)

In [33]:
# 출판사
publisher = book_list[0].select_one(".authPub.info_pub > a").text

In [34]:
# 출간일
pub_date = book_list[0].select_one(".authPub.info_date").text

In [35]:
# 책 가격
price = int(book_list[0].select_one(".yes_b").text.replace(",", ""))

In [36]:
# 평점
rating = float(book_list[0].select_one(".rating_grade > .yes_b").text)

In [37]:
# 리뷰수
n_reviews = int(book_list[0].select_one(".rating_rvCount > a > em.txC_blue").text)

In [38]:
print(book_title, author, publisher, pub_date,  price, rating, n_reviews)

최소한의 삼국지 태수 프런트페이지 2025년 11월 17550 10.0 42


# 페이지 전체에 있는 책 정보 하나로 모으기


In [39]:
result = {}
keys = ['제목', '저자', '출판사', '출간일', '가격', '평점', '리뷰수']
for idx ,book in enumerate(book_list):
    print(f"{idx+1}/{len(book_list)}")
    book_title = book.select_one(".gd_name").text
    author = text_clean(book.select_one(".authPub.info_auth").text)
    publisher = book.select_one(".authPub.info_pub > a").text
    pub_date = book.select_one(".authPub.info_date").text
    price = int(book.select_one(".yes_b").text.replace(",", ""))
    rating = float(book.select_one(".rating_grade > .yes_b").text) if book.select_one(".rating_grade > .yes_b") != None else 0
    n_reviews = int(book.select_one(".rating_rvCount > a > em.txC_blue").text.replace(",", "")) if book.select_one(".rating_rvCount > a > em.txC_blue") != None else 0
    values = (book_title, author, publisher, pub_date,  price, rating, n_reviews)
    for key, value in zip(keys, values):
        result.setdefault(key, []).append(value)

df = pd.DataFrame(result)
df

1/120
2/120
3/120
4/120
5/120
6/120
7/120
8/120
9/120
10/120
11/120
12/120
13/120
14/120
15/120
16/120
17/120
18/120
19/120
20/120
21/120
22/120
23/120
24/120
25/120
26/120
27/120
28/120
29/120
30/120
31/120
32/120
33/120
34/120
35/120
36/120
37/120
38/120
39/120
40/120
41/120
42/120
43/120
44/120
45/120
46/120
47/120
48/120
49/120
50/120
51/120
52/120
53/120
54/120
55/120
56/120
57/120
58/120
59/120
60/120
61/120
62/120
63/120
64/120
65/120
66/120
67/120
68/120
69/120
70/120
71/120
72/120
73/120
74/120
75/120
76/120
77/120
78/120
79/120
80/120
81/120
82/120
83/120
84/120
85/120
86/120
87/120
88/120
89/120
90/120
91/120
92/120
93/120
94/120
95/120
96/120
97/120
98/120
99/120
100/120
101/120
102/120
103/120
104/120
105/120
106/120
107/120
108/120
109/120
110/120
111/120
112/120
113/120
114/120
115/120
116/120
117/120
118/120
119/120
120/120


,제목,저자,출판사,출간일,가격,평점,리뷰수
0,최소한의 삼국지,최태성 /이성원 감수,프런트페이지,2025년 11월,17550,10.0,42
1,트렌드 코리아 2026,김난도 전미영 최지혜 권정윤 한다혜 외 7명 ...,미래의창,2025년 09월,18000,9.2,164
2,어른의 행복은 조용하다,태수,페이지2북스,2024년 11월,16020,9.5,405
3,혼모노,성해나,창비,2025년 03월,16200,9.1,688
4,박곰희 연금 부자 수업 (10만 부 기념 스페셜 에디션),박곰희,인플루엔셜,2025년 06월,18900,9.7,189
...,...,...,...,...,...,...,...
115,정승제의 수학 대모험 1,설민석 조영선 글/최진규 박지영 그림/정승제 감수,단꿈아이,2025년 11월,14400,9.9,137
116,슈뻘맨의 숨은 과학 찾기 7,슈뻘맨 원/서후 글/류수형 그림/샌드박스 네트워크 정재형 감수,미래엔아이세움,2025년 11월,15120,10.0,5
117,마일리지 아워,최유나,북로망스,2025년 11월,19800,9.9,27
118,세이노의 가르침,세이노(SayNo),데이원,2023년 03월,6480,9.0,2802


1. 베스트셀러 전체 책 가져오기
2. 저자/역자 구분하기
3. 저자가 여러명일 경우 저자 모두 추출해서 가져오기
4. 원저, 그림, 역자, 감수가 있을 경우에도 모두 분리하기
5. 책 상세페이지 링크도 수집
------------------------------------------------------------
1. 상세페이지로 들어가 품목 정보에서 페이지수 추출
2. 카테고리 분류 내용 가져오기
3. 책소개 글 가져오기

In [40]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup as bs

In [41]:
author_list = []

for page in range(1,10):
    url = "https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = bs(r.content, 'lxml')
    book_list = response.select("#yesBestList > li")
    print(f"{page+1}/9 페이지 수집중                 ")
    
    for idx, book in enumerate(book_list):
        print(f"{idx+1}/{len(book_list)} 자료 수집중", end="\r")
        result = dict(책제목="", 저 = "", 원저 = "", 글 = "", 역 = "", 글그림 = "", 사진 = "", 감수="",
                            그림 = "", 공저 = "", 기획="")
        result['책제목'] = book.select_one(".gd_name").text
        # 저자추출
        if "/" not in book.select_one(".authPub.info_auth").text:
            result['저'] = text_clean(book.select_one(".authPub.info_auth").text) 
        elif "/" in book.select_one(".authPub.info_auth").text:
            n_item = len(book.select_one(".authPub.info_auth").text.split("/"))
            items = book.select_one(".authPub.info_auth").text.split("/")
    #         print(idx, n_item, items, end="\n\n")
            if n_item == 2 and "정보 더 보기" in items[0]:
                result['저'] = text_clean(book.select_one(".moreAuthLiCont").text.replace("\n", " "))
            elif n_item >= 2 and "정보 더 보기" not in items[0]:
                for item in items:
    #                 print(item.split()[-1])
                    author_key = item.split()[-1]
                    result[author_key] = " ".join(item.split()[:-1])


        result['출판사'] = book.select_one(".authPub.info_pub > a").text
        result['출간일'] = book.select_one(".authPub.info_date").text
        result['가격'] = int(book.select_one(".yes_b").text.replace(",", ""))
        result['평점'] = float(book.select_one(".rating_grade > .yes_b").text) if book.select_one(".rating_grade > .yes_b") != None else 0
        result['리뷰수'] = int(text_clean(book.select_one(".rating_rvCount > a > em.txC_blue").text)) if book.select_one(".rating_rvCount > a > em.txC_blue") != None else 0
        result['상세링크'] = "https://www.yes24.com"+ book.select_one(".info_row.info_name > .gd_name")['href']

        author_list.append(result)
    
    time.sleep(3)
    
    
    
df = pd.DataFrame(author_list)
df

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
2/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=2&pageSize=120
200
3/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=3&pageSize=120
200
4/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=4&pageSize=120
200
5/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=5&pageSize=120
200
6/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=6&pageSize=120
200
7/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=7&pageSize=120
200
8/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=8&pageSize=120
200
9/9 페

,책제목,저,원저,글,역,글그림,사진,감수,그림,공저,...,평점,리뷰수,상세링크,편,등저,편저,해설,편역,보기,이정모
0,최소한의 삼국지,최태성,,,,,,이성원,,,...,10.0,42,https://www.yes24.com/product/goods/166390503,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,트렌드 코리아 2026,김난도 전미영 최지혜 권정윤 한다혜 이혜원 이수진 서유현 전다현 이준영 이향은 김나은,,,,,,,,,...,9.2,164,https://www.yes24.com/product/goods/153064968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,어른의 행복은 조용하다,태수,,,,,,,,,...,9.5,405,https://www.yes24.com/product/goods/136298166,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,혼모노,성해나,,,,,,,,,...,9.1,688,https://www.yes24.com/product/goods/143911524,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,박곰희 연금 부자 수업 (10만 부 기념 스페셜 에디션),박곰희,,,,,,,,,...,9.7,189,https://www.yes24.com/product/goods/148032819,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,최고의 주식 최적의 타이밍,윌리엄 J. 오닐,,,,,,,,,...,9.2,171,https://www.yes24.com/product/goods/7240183,NaN,NaN,NaN,NaN,NaN,NaN,NaN
996,불꽃 수영 대회,,,신현경,,,,,노예지,,...,9.9,151,https://www.yes24.com/product/goods/141180543,NaN,NaN,NaN,NaN,NaN,NaN,NaN
997,"돈, 뜨겁게 사랑하고 차갑게 다루어라",앙드레 코스톨라니,,,한윤진,,,,,,...,10.0,1,https://www.yes24.com/product/goods/167475858,NaN,NaN,NaN,NaN,NaN,NaN,NaN
998,제1회 안타까운 동물 자랑 대회,,,,이선희,,,,"시모마 아야에, 가와무라 후유미, 도쿠나가 아키코",,...,9.9,25,https://www.yes24.com/product/goods/160088937,NaN,NaN,NaN,NaN,NaN,이마이즈미 타다아키 감수 외 1명 정보 더,감추기 시모마 아야에 가와무라 후유미 도쿠나가 아키코 이선희 이마이즈미 타다아키


# 상세페이지 수집하기

In [42]:
def detail_extraction(url):
    r =requests.get(url)
    soup = bs(r.content, 'lxml')
    book_page = soup.select_one(".infoSetCont_wrap > div > table > tbody > tr:nth-child(2) > td").text.split()[0][:-1] if soup.select_one(".infoSetCont_wrap > div > table > tbody > tr:nth-child(2) > td") != None else 0
    category = text_clean(soup.select_one("div.infoSetCont_wrap > dl:nth-child(1) > dd > ul").text).replace(">", " ").replace("/", " ") if soup.select_one("div.infoSetCont_wrap > dl:nth-child(1) > dd > ul") != None else "카테고리 없음"
    description = text_clean(soup.select_one(".infoWrap_txtInner").text) if soup.select_one(".infoWrap_txtInner") != None else "책소개 없음"
    return book_page, category, description

In [43]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup as bs

In [44]:
author_list = []

for page in range(1,10):
    url = "https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = bs(r.content, 'lxml')
    book_list = response.select("#yesBestList > li")
    print(f"{page+1}/9 페이지 수집중                 ")
    
    for idx, book in enumerate(book_list):
        print(f"{idx+1}/{len(book_list)} 자료 수집중", end="\r")
        result = dict(책제목="", 저 = "", 원저 = "", 글 = "", 역 = "", 글그림 = "", 사진 = "", 감수="",
                            그림 = "", 공저 = "", 기획="")
        result['책제목'] = book.select_one(".gd_name").text
        # 저자추출
        if "/" not in book.select_one(".authPub.info_auth").text:
            result['저'] = text_clean(book.select_one(".authPub.info_auth").text) 
        elif "/" in book.select_one(".authPub.info_auth").text:
            n_item = len(book.select_one(".authPub.info_auth").text.split("/"))
            items = book.select_one(".authPub.info_auth").text.split("/")
    #         print(idx, n_item, items, end="\n\n")
            if n_item == 2 and "정보 더 보기" in items[0]:
                result['저'] = text_clean(book.select_one(".moreAuthLiCont").text.replace("\n", " "))
            elif n_item >= 2 and "정보 더 보기" not in items[0]:
                for item in items:
    #                 print(item.split()[-1])
                    author_key = item.split()[-1]
                    result[author_key] = " ".join(item.split()[:-1])


        result['출판사'] = book.select_one(".authPub.info_pub > a").text
        result['출간일'] = book.select_one(".authPub.info_date").text
        result['가격'] = int(book.select_one(".yes_b").text.replace(",", ""))
        result['평점'] = float(book.select_one(".rating_grade > .yes_b").text) if book.select_one(".rating_grade > .yes_b") != None else 0
        result['리뷰수'] = int(text_clean(book.select_one(".rating_rvCount > a > em.txC_blue").text)) if book.select_one(".rating_rvCount > a > em.txC_blue") != None else 0
        result['상세링크'] = "https://www.yes24.com"+ book.select_one(".info_row.info_name > .gd_name")['href']
        book_page, category, description = detail_extraction(result['상세링크'])
        result['페이지수'] = book_page
        result['카테고리'] = category
        result['description'] = description
        
        author_list.append(result)
    
    time.sleep(3)
    
    
    
df = pd.DataFrame(author_list)
df

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
2/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=2&pageSize=120
200
3/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=3&pageSize=120
200
4/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=4&pageSize=120
200
5/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=5&pageSize=120
200
6/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=6&pageSize=120
200
7/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=7&pageSize=120
200
8/9 페이지 수집중                 
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=8&pageSize=120
200
9/9 페

,책제목,저,원저,글,역,글그림,사진,감수,그림,공저,...,페이지수,카테고리,description,편,등저,편저,해설,편역,보기,이정모
0,최소한의 삼국지,최태성,,,,,,이성원,,,...,352,국내도서 인문 인문 교양 교양으로 읽는 인문 국내도서 인문 독서 비평 고전 읽기 ...,★★★ 대한민국 대표 지식 스토리텔러 ★★★★★★ 700만이 선택한 명강사 신간 ★...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,트렌드 코리아 2026,김난도 전미영 최지혜 권정윤 한다혜 이혜원 이수진 서유현 전다현 이준영 이향은 김나은,,,,,,,,,...,400,국내도서 경제 경영 마케팅 세일즈 트렌드 미래예측,HORSE POWERAI 대전환의 시대 무엇을 준비해야 하는가? 세상은 작용과 반작...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,어른의 행복은 조용하다,태수,,,,,,,,,...,288,국내도서 에세이 삶의 자세와 지혜 국내도서 에세이 한국 에세이,행복을 찾는 방법이 아니라불행에 대한 수비력을 길러주는58가지 인생 이야기《1cm ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,혼모노,성해나,,,,,,,,,...,368,국내도서 소설 시 희곡 한국소설 한국 단편소설,“‘몰입’의 파티다. 영화로 만들고 싶은 작품들로 가득하다.” -배우 박정민‘202...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,박곰희 연금 부자 수업 (10만 부 기념 스페셜 에디션),박곰희,,,,,,,,,...,300,국내도서 경제 경영 투자 재테크 재테크일반,“연금 투자야말로 평범한 사람들이 부자가 되는 확실한 방법이다!”출간 6개월 만에 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,최고의 주식 최적의 타이밍,윌리엄 J. 오닐,,,,,,,,,...,432,국내도서 경제 경영 투자 재테크 주식 증권,1988년 미국에서 처음 출간되자마자 성공 투자의 비결을 너무나도 정확하고 냉정하게...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
996,불꽃 수영 대회,,,신현경,,,,,노예지,,...,양,국내도서 어린이 1-2학년 1-2학년 그림 동화책 1-2학년 창작동화 국내도서 어...,뜨거운 인기! 28주 연속 서점 베스트셀러 〈야옹이 수영 교실〉 시리즈 3권 출간2...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
997,"돈, 뜨겁게 사랑하고 차갑게 다루어라",앙드레 코스톨라니,,,한윤진,,,,,,...,양,국내도서 경제 경영 투자 재테크 주식 증권,앙드레 코스톨라니 투자총서 시리즈 25주년 기념양장본 특별판 출시 - 3종 한정 ‘...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
998,제1회 안타까운 동물 자랑 대회,,,,이선희,,,,"시모마 아야에, 가와무라 후유미, 도쿠나가 아키코",,...,176,국내도서 어린이 3-4학년 3-4학년 학습 3-4학년 과학 환경 국내도서 어린이 ...,일본 530만 부 초베스트셀러 동물도감!자랑할수록 안타깝기만 한 동물들의 진화 이야...,NaN,NaN,NaN,NaN,NaN,이마이즈미 타다아키 감수 외 1명 정보 더,감추기 시모마 아야에 가와무라 후유미 도쿠나가 아키코 이선희 이마이즈미 타다아키
